# EDA — Indian Weather (T024)

Notebook de **análise exploratória** para o marco M02 / história S02. Restrição da equipa: **PyArrow** + **Matplotlib** + **NumPy** (sem pandas nem scikit-learn).

## Amostra e reprodutibilidade

A primeira célula de código define `N_AMOSTRA` (linhas lidas do Parquet após seleção de colunas). **Gráficos e estatísticas descritas aqui referem-se a essa amostra**, não necessariamente ao ficheiro completo, salvo indicação em contrário.

## Documentação relacionada

- Alvo e pipeline: [preprocessamento.md](../docs/preprocessamento.md)
- Desequilíbrio de classes: [imbalance.md](../docs/imbalance.md)


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Tamanho da amostra (linhas) após leitura com colunas selecionadas
N_AMOSTRA = 20000000


def col_to_numpy(col, *, dtype=None):
    """Compat: PyArrow Array/ChunkedArray -> numpy.

    Algumas versões não aceitam kwargs em `to_numpy()`. Para ChunkedArray,
    combinamos chunks antes para evitar surpresas.
    """

    if hasattr(col, "combine_chunks"):
        try:
            col = col.combine_chunks()
        except Exception:
            pass

    try:
        arr = col.to_numpy(zero_copy_only=False)
    except TypeError:
        arr = col.to_numpy()

    if dtype is not None:
        return np.asarray(arr, dtype=dtype)
    return np.asarray(arr)


def resolve_repo_and_fig_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    roots = [cwd]
    if cwd.name == "notebooks":
        roots.append(cwd.parent)
    for root in roots:
        if (root / "requirements.txt").exists():
            fig = root / "notebooks" / "figuras"
            fig.mkdir(parents=True, exist_ok=True)
            return root, fig
    fig = cwd / "figuras"
    fig.mkdir(parents=True, exist_ok=True)
    return cwd, fig


REPO_ROOT, FIG_DIR = resolve_repo_and_fig_dir()
PARQUET = REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet"

os.environ.setdefault("MPLBACKEND", "Agg")

COLS_LEITURA = [
    "datetime",
    "rain_label",
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
    "lat",
    "lon",
    "hour",
    "month",
]

if not PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {PARQUET}. Coloque o dataset em data/ e execute a partir da raiz do repositório ou abra o Jupyter com cwd na raiz."
    )

table = pq.read_table(PARQUET, columns=COLS_LEITURA, use_threads=True)
n_full = table.num_rows
if table.num_rows > N_AMOSTRA:
    table = table.slice(0, N_AMOSTRA)

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", FIG_DIR)
print("Linhas no ficheiro (aprox.):", n_full)
print("Linhas usadas neste notebook:", table.num_rows)
print(table.schema)


REPO_ROOT: /home/jovyan/work
FIG_DIR: /home/jovyan/work/notebooks/figuras
Linhas no ficheiro (aprox.): 46082160
Linhas usadas neste notebook: 20000000
datetime: timestamp[ms]
rain_label: int64
temperature_C: double
humidity_pct: int64
precip_mm: double
cloud_cover_pct: int64
pressure_hPa: double
dew_point_C: double
wind_speed_ms: double
solar_radiation_Wm2: double
lat: double
lon: double
hour: int64
month: int64


## Distribuição do alvo (`temperature_C`)

Como vamos tratar **regressão**, o foco é entender a faixa típica de temperatura, caudas/outliers e possíveis recortes (hora/mês/local). Aqui olhamos a distribuição na amostra, com estatísticas descritivas e percentis.


In [2]:

t = col_to_numpy(table.column("temperature_C"), dtype=np.float64)

t_valid = t[~np.isnan(t)]

stats = {
    "n_total": int(t.shape[0]),
    "n_valid": int(t_valid.shape[0]),
    "n_nan": int(np.isnan(t).sum()),
    "min": float(np.nanmin(t)),
    "p01": float(np.nanpercentile(t_valid, 1)),
    "p05": float(np.nanpercentile(t_valid, 5)),
    "p25": float(np.nanpercentile(t_valid, 25)),
    "median": float(np.nanpercentile(t_valid, 50)),
    "p75": float(np.nanpercentile(t_valid, 75)),
    "p95": float(np.nanpercentile(t_valid, 95)),
    "p99": float(np.nanpercentile(t_valid, 99)),
    "max": float(np.nanmax(t)),
    "mean": float(np.nanmean(t)),
    "std": float(np.nanstd(t)),
}
print("Resumo temperature_C (amostra):")
for k in [
    "n_total",
    "n_valid",
    "n_nan",
    "min",
    "p01",
    "p05",
    "p25",
    "median",
    "p75",
    "p95",
    "p99",
    "max",
    "mean",
    "std",
]:
    print(f"- {k}: {stats[k]}")

# Histograma (bins por Freedman–Diaconis com fallback)
q75, q25 = np.nanpercentile(t_valid, [75, 25])
iqr = q75 - q25
if iqr > 0:
    bin_w = 2 * iqr * (t_valid.shape[0] ** (-1 / 3))
    bins = int(np.clip(np.ceil((np.nanmax(t_valid) - np.nanmin(t_valid)) / max(bin_w, 1e-9)), 30, 160))
else:
    bins = 60

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.hist(t_valid, bins=bins, color="#4C72B0", edgecolor="#1f2d3d", linewidth=0.4)
ax.set_xlabel("temperature_C")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de temperature_C (amostra)")
ax.grid(True, axis="y", alpha=0.25)

msg = (
    f"n={stats['n_valid']:,}\\n"
    f"p05={stats['p05']:.2f} | p50={stats['median']:.2f} | p95={stats['p95']:.2f}\\n"
    f"mean={stats['mean']:.2f} ± {stats['std']:.2f}"
)
ax.text(
    0.98,
    0.98,
    msg,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9),
)

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_distribuicao.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_distribuicao.png")


Resumo temperature_C (amostra):
- n_total: 20000000
- n_valid: 20000000
- n_nan: 0
- min: -18.5
- p01: 3.5
- p05: 9.8
- p25: 18.8
- median: 24.4
- p75: 28.1
- p95: 34.7
- p99: 39.5
- max: 48.1
- mean: 23.351607475000026
- std: 7.451051793540566
Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_distribuicao.png


## Relações feature–alvo (regressão em `temperature_C`)

Aqui a ideia é ver como a temperatura muda com:

- `hour` e `month` (sazonalidade)
- variáveis físicas diretamente relacionadas (`dew_point_C`, `humidity_pct`, `pressure_hPa`, etc.)

Vamos começar com **médias de `temperature_C` por `hour`/`month`** e depois um **scatter** com subsample para enxergar relação com `dew_point_C` e `humidity_pct`.


In [3]:
g_h = table.group_by("hour").aggregate([
    ("temperature_C", "mean"),
    ("temperature_C", "count"),
])
g_m = table.group_by("month").aggregate([
    ("temperature_C", "mean"),
    ("temperature_C", "count"),
])

# Ordenação (hour 0..23, month 1..12)
hour_vals = col_to_numpy(g_h.column("hour"), dtype=np.int64)
order_h = np.argsort(hour_vals)
h_x = hour_vals[order_h]
h_mean = col_to_numpy(g_h.column("temperature_C_mean"), dtype=np.float64)[order_h]
h_cnt = col_to_numpy(g_h.column("temperature_C_count"), dtype=np.float64)[order_h]

month_vals = col_to_numpy(g_m.column("month"), dtype=np.int64)
order_m = np.argsort(month_vals)
m_x = month_vals[order_m]
m_mean = col_to_numpy(g_m.column("temperature_C_mean"), dtype=np.float64)[order_m]
m_cnt = col_to_numpy(g_m.column("temperature_C_count"), dtype=np.float64)[order_m]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(h_x, h_mean, color="#4C72B0", linewidth=1.8)
ax1.scatter(h_x, h_mean, s=22, color="#2b4c7e")
ax1.set_xlabel("hour")
ax1.set_ylabel("média temperature_C")
ax1.set_title("Média de temperature_C por hora (amostra)")
ax1.set_xticks(np.arange(0, 24, 2))
ax1.grid(True, axis="y", alpha=0.25)

ax2.plot(m_x, m_mean, color="#C44E52", linewidth=1.8)
ax2.scatter(m_x, m_mean, s=22, color="#7c2c2f")
ax2.set_xlabel("month")
ax2.set_ylabel("média temperature_C")
ax2.set_title("Média de temperature_C por mês (amostra)")
ax2.set_xticks(np.arange(1, 13, 1))
ax2.grid(True, axis="y", alpha=0.25)

fig.suptitle("Sazonalidade simples do alvo — médias condicionais", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_por_hour_month.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_por_hour_month.png")

print("Resumo group_by hour:")
print("hours:", h_x.tolist())
print("counts:", h_cnt.astype(int).tolist())
print("Resumo group_by month:")
print("months:", m_x.tolist())
print("counts:", m_cnt.astype(int).tolist())


Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_por_hour_month.png
Resumo group_by hour:
hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
counts: [833334, 833334, 833334, 833334, 833334, 833334, 833334, 833334, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333, 833333]
Resumo group_by month:
months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
counts: [1741704, 1587648, 1718304, 1631520, 1685904, 1631520, 1685904, 1685576, 1630800, 1685160, 1630800, 1685160]


In [4]:
# Relação com variáveis físicas (subsample para visualização)
# Observação: dew_point_C é termodinamicamente ligado a temperature_C → correlação alta é esperada.

rng = np.random.default_rng(42)

temp = col_to_numpy(table.column("temperature_C"), dtype=np.float64)
dew = col_to_numpy(table.column("dew_point_C"), dtype=np.float64)
hum = col_to_numpy(table.column("humidity_pct"), dtype=np.float64)
pres = col_to_numpy(table.column("pressure_hPa"), dtype=np.float64)

valid = (~np.isnan(temp)) & (~np.isnan(dew)) & (~np.isnan(hum)) & (~np.isnan(pres))
idx = np.flatnonzero(valid)

n_plot = min(60_000, idx.shape[0])
if n_plot == 0:
    raise ValueError("Sem dados válidos suficientes para scatter (temperature/dew/humidity/pressure).")

pick = rng.choice(idx, size=n_plot, replace=False)

temp_s = temp[pick]
dew_s = dew[pick]
hum_s = hum[pick]
pres_s = pres[pick]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

axes[0].scatter(dew_s, temp_s, s=6, alpha=0.22, color="#4C72B0", edgecolors="none")
axes[0].set_xlabel("dew_point_C")
axes[0].set_ylabel("temperature_C")
axes[0].set_title("temperature_C vs dew_point_C")
axes[0].grid(True, alpha=0.25)

axes[1].scatter(hum_s, temp_s, s=6, alpha=0.22, color="#55A868", edgecolors="none")
axes[1].set_xlabel("humidity_pct")
axes[1].set_ylabel("temperature_C")
axes[1].set_title("temperature_C vs humidity_pct")
axes[1].grid(True, alpha=0.25)

axes[2].scatter(pres_s, temp_s, s=6, alpha=0.22, color="#C44E52", edgecolors="none")
axes[2].set_xlabel("pressure_hPa")
axes[2].set_ylabel("temperature_C")
axes[2].set_title("temperature_C vs pressure_hPa")
axes[2].grid(True, alpha=0.25)

fig.suptitle("Relações alvo–preditores (subsample; amostra)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_temperature_scatter_relacoes.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_temperature_scatter_relacoes.png")

# Correlações rápidas no subsample (sinal exploratório)
print("Corr(subsample) temperature vs dew_point:", float(np.corrcoef(temp_s, dew_s)[0, 1]))
print("Corr(subsample) temperature vs humidity:", float(np.corrcoef(temp_s, hum_s)[0, 1]))
print("Corr(subsample) temperature vs pressure:", float(np.corrcoef(temp_s, pres_s)[0, 1]))


Figura: /home/jovyan/work/notebooks/figuras/eda_temperature_scatter_relacoes.png
Corr(subsample) temperature vs dew_point: 0.49728419020328246
Corr(subsample) temperature vs humidity: -0.4480269474366121
Corr(subsample) temperature vs pressure: 0.43167929016072604


## Correlação entre preditores (amostra)

Subconjunto de colunas numéricas alinhadas a [preprocessamento.md](../docs/preprocessamento.md). A matriz usa `numpy.corrcoef`; valores ausentes são substituídos temporariamente pela média da coluna **só para este cálculo exploratório** (não substitui decisões do pipeline T022).


In [5]:
corr_cols = [
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
]
rows = []
for c in corr_cols:
    v = col_to_numpy(table.column(c), dtype=np.float64)
    rows.append(v)
X = np.vstack(rows)
col_means = np.nanmean(X, axis=1, keepdims=True)
X_filled = np.where(np.isnan(X), col_means, X)
C = np.corrcoef(X_filled)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Correlação de Pearson — amostra; NaN imputados pela média da coluna")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_correlacao_preditoras.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_correlacao_preditoras.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_correlacao_preditoras.png


In [6]:
# Ranking numérico das correlações (com base na matriz C já calculada)
# C e corr_cols vêm da célula anterior.

pairs = []
n = len(corr_cols)
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((corr_cols[i], corr_cols[j], float(C[i, j])))

# Top por magnitude absoluta
pairs_abs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
# Top positivas e negativas
pairs_pos = sorted(pairs, key=lambda x: x[2], reverse=True)
pairs_neg = sorted(pairs, key=lambda x: x[2])

TOP_K = 10

print(f"Top {TOP_K} por |correlação|:")
for a, b, v in pairs_abs[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

print(f"\nTop {TOP_K} correlações positivas:")
for a, b, v in pairs_pos[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

print(f"\nTop {TOP_K} correlações negativas:")
for a, b, v in pairs_neg[:TOP_K]:
    print(f"- {a}  x  {b}: {v:+.4f}")

# Extra: ranking focado no alvo de regressão
target = "temperature_C"
with_target = [(a, b, v) for a, b, v in pairs if a == target or b == target]
with_target = sorted(with_target, key=lambda x: abs(x[2]), reverse=True)

print(f"\nCorrelações com {target} (ordenadas por |corr|):")
for a, b, v in with_target:
    other = b if a == target else a
    print(f"- {target}  x  {other}: {v:+.4f}")


Top 10 por |correlação|:
- humidity_pct  x  dew_point_C: +0.5164
- humidity_pct  x  solar_radiation_Wm2: -0.5146
- temperature_C  x  dew_point_C: +0.5003
- temperature_C  x  solar_radiation_Wm2: +0.4582
- humidity_pct  x  cloud_cover_pct: +0.4503
- temperature_C  x  humidity_pct: -0.4470
- cloud_cover_pct  x  dew_point_C: +0.4369
- temperature_C  x  pressure_hPa: +0.4340
- temperature_C  x  wind_speed_ms: +0.3345
- pressure_hPa  x  dew_point_C: +0.3052

Top 10 correlações positivas:
- humidity_pct  x  dew_point_C: +0.5164
- temperature_C  x  dew_point_C: +0.5003
- temperature_C  x  solar_radiation_Wm2: +0.4582
- humidity_pct  x  cloud_cover_pct: +0.4503
- cloud_cover_pct  x  dew_point_C: +0.4369
- temperature_C  x  pressure_hPa: +0.4340
- temperature_C  x  wind_speed_ms: +0.3345
- pressure_hPa  x  dew_point_C: +0.3052
- pressure_hPa  x  wind_speed_ms: +0.2709
- precip_mm  x  cloud_cover_pct: +0.2592

Top 10 correlações negativas:
- humidity_pct  x  solar_radiation_Wm2: -0.5146
- temper

## Dimensão temporal (`hour`, `month`, `datetime`)

Exploração de sazonalidade horária e mensal e de contagem de registos por dia (data truncada a partir de `datetime`).

As contagens por dia usam **apenas as primeiras `N_AMOSTRA` linhas** do Parquet, na ordem em que o ficheiro foi lido; se essa fatia for temporalmente contígua e com cadência fixa por dia, o gráfico pode aparecer em **patamares** — isso reflecte a estrutura da amostra, não um erro de ordenação (os dias são ordenados antes do plot).


In [7]:
h_raw = col_to_numpy(table.column("hour"), dtype=np.float64)
m_raw = col_to_numpy(table.column("month"), dtype=np.float64)
h = h_raw[~np.isnan(h_raw)].astype(np.int64, copy=False)
m = m_raw[~np.isnan(m_raw)].astype(np.int64, copy=False)
h = np.clip(h, 0, 23)
m = np.clip(m, 1, 12)
c_h = np.bincount(h, minlength=24)
c_m = np.bincount(m - 1, minlength=12)
hours_x = np.arange(24)
months_x = np.arange(1, 13)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(hours_x, c_h, color="#55A868", edgecolor="#1a3d22", linewidth=0.65)
ax1.set_xlabel("hour (0–23)")
ax1.set_ylabel("Contagem")
ax1.set_title("Contagem por hora (amostra)")
ax1.set_xticks(np.arange(0, 24, 2))
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(months_x, c_m, color="#C44E52", edgecolor="#4a1518", linewidth=0.65)
ax2.set_xlabel("month (1–12)")
ax2.set_ylabel("Contagem")
ax2.set_title("Contagem por mês (amostra)")
ax2.set_xticks(months_x)
ax2.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_hour_month_histogramas.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_hour_month_histogramas.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_hour_month_histogramas.png


In [8]:
d_arr = pc.cast(table.column("datetime"), pa.date32())
st = pc.value_counts(d_arr)
days = col_to_numpy(st.field(0), dtype=np.int64)
cnts = col_to_numpy(st.field(1), dtype=np.int64)
order = np.argsort(days)
days_s = days[order]
cnts_s = cnts[order].astype(float)
x = np.asarray(days_s, dtype="datetime64[D]")

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.step(x, cnts_s, where="post", color="#4C72B0", linewidth=1.1, label="contagem/dia")
ax.scatter(x, cnts_s, s=10, color="#2b4c7e", alpha=0.55, zorder=3)
ax.set_xlabel("Data")
ax.set_ylabel("Nº de linhas por dia (só na amostra)")
ax.set_title("Contagem por dia — fatia inicial de N linhas do Parquet")
ax.grid(True, axis="y", alpha=0.35)
ax.legend(loc="upper right", fontsize=8)
fig.autofmt_xdate()
fig.text(
    0.5,
    0.01,
    "Patamares = cadência estável por dia nesta fatia; dias ordenados antes do gráfico.",
    ha="center",
    fontsize=8,
    transform=fig.transFigure,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(FIG_DIR / "eda_contagem_por_dia.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_contagem_por_dia.png")


Figura: /home/jovyan/work/notebooks/figuras/eda_contagem_por_dia.png


## Insights acionáveis (regressão em `temperature_C`)

1. **Faixa do alvo e outliers:** usar percentis (p05–p95) para entender caudas e evitar decisões baseadas em extremos raros. Se houver valores fisicamente improváveis, vale checar qualidade de dados antes de modelar.

2. **Sazonalidade:** `hour` e `month` devem explicar parte relevante do sinal; no pipeline, considerar representação **cíclica** (sin/cos) em vez de ordinal puro, e fazer **split temporal** para não "vazar" padrões.

3. **Preditores termodinâmicos:** `dew_point_C` tende a ser altamente informativo para `temperature_C`. Se o objetivo é previsão operacional, ok; se o objetivo é entender causalidade/impacto de outras variáveis, essa redundância pode "dominar" o modelo.

4. **Avaliação:** para regressão, preferir MAE/RMSE e métricas por fatias (por mês/hora/região) para garantir desempenho consistente ao longo do tempo e espaço.

5. **Completude:** verificar quais colunas têm valores ausentes sistemáticos. Decidir entre imputação, drop, ou estratificação por completude.

6. **Variação geoespacial:** se há clustering significativo ou variação regional de temperatura, considerar adicionar features de localização (lat/lon binadas ou embedding) ou estratificar validação.

7. **Rain_label e precip_mm:** entender se `rain_label` é alvo auxiliar, feature, ou apenas diagnóstico. Verificar consistência com `precip_mm`; se desbalanceada, estratificar split.

8. **Qualidade de solar_radiation:** ciclo claro dia/noite é esperado. Se houver anomalias (ex: radiação alta à noite), flags de qualidade podem ser úteis.

9. **Autocorrelação temporal:** altos valores em lags curtos (1-24h) justificam features de lag. Isso também reforça importância de **split temporal** e validação em ordem cronológica (não aleatória).

10. **Próximo passo:** materializar splits e pipeline em Spark com **fit apenas no treino**, e validar generalização na janela temporal de validação/teste.


## Completude de dados (valores ausentes por coluna)

Análise do padrão de NaN em todas as colunas. Importante para decisões de imputação e estratificação no pipeline.


In [9]:
completeness = {}
for col_name in COLS_LEITURA:
    col = table.column(col_name)
    n_total = col.length()
    n_null = pc.sum(pc.is_null(col)).as_py()
    pct_missing = 100.0 * n_null / n_total if n_total > 0 else 0
    completeness[col_name] = {
        "n_total": n_total,
        "n_null": n_null,
        "n_valid": n_total - n_null,
        "pct_missing": pct_missing,
    }

print("Completude por coluna (amostra):")
print("-" * 70)
for col_name in COLS_LEITURA:
    c = completeness[col_name]
    print(f"{col_name:25s} | valid: {c['n_valid']:>8,d} | missing: {c['n_null']:>6,d} ({c['pct_missing']:>5.2f}%)")

fig, ax = plt.subplots(figsize=(10, 4.5))
cols_sorted = sorted(COLS_LEITURA, key=lambda x: completeness[x]["pct_missing"], reverse=True)
pcts = [completeness[c]["pct_missing"] for c in cols_sorted]
colors = ["#C44E52" if p > 0 else "#55A868" for p in pcts]

ax.barh(cols_sorted, pcts, color=colors, edgecolor="#1f2d3d", linewidth=0.6)
ax.set_xlabel("Percentagem ausente (%)")
ax.set_title("Completude de dados por coluna (amostra)")
ax.grid(True, axis="x", alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_completude_dados.png", dpi=150)
plt.close(fig)
print("\nFigura:", FIG_DIR / "eda_completude_dados.png")


Completude por coluna (amostra):
----------------------------------------------------------------------
datetime                  | valid: 20,000,000 | missing:      0 ( 0.00%)
rain_label                | valid: 20,000,000 | missing:      0 ( 0.00%)
temperature_C             | valid: 20,000,000 | missing:      0 ( 0.00%)
humidity_pct              | valid: 20,000,000 | missing:      0 ( 0.00%)
precip_mm                 | valid: 20,000,000 | missing:      0 ( 0.00%)
cloud_cover_pct           | valid: 20,000,000 | missing:      0 ( 0.00%)
pressure_hPa              | valid: 20,000,000 | missing:      0 ( 0.00%)
dew_point_C               | valid: 20,000,000 | missing:      0 ( 0.00%)
wind_speed_ms             | valid: 20,000,000 | missing:      0 ( 0.00%)
solar_radiation_Wm2       | valid: 20,000,000 | missing:      0 ( 0.00%)
lat                       | valid: 20,000,000 | missing:      0 ( 0.00%)
lon                       | valid: 20,000,000 | missing:      0 ( 0.00%)
hour                

## Exploração geoespacial (lat/lon)

Distribuição espacial dos dados. Verifica clustering geográfico e variabilidade de temperatura por região.


In [10]:
lat = col_to_numpy(table.column("lat"), dtype=np.float64)
lon = col_to_numpy(table.column("lon"), dtype=np.float64)
temp = col_to_numpy(table.column("temperature_C"), dtype=np.float64)

valid_geo = (~np.isnan(lat)) & (~np.isnan(lon)) & (~np.isnan(temp))
lat_v = lat[valid_geo]
lon_v = lon[valid_geo]
temp_v = temp[valid_geo]

print("Estatísticas geoespaciais:")
print(f"- Latitude: min={np.min(lat_v):.4f}, max={np.max(lat_v):.4f}, mean={np.mean(lat_v):.4f}")
print(f"- Longitude: min={np.min(lon_v):.4f}, max={np.max(lon_v):.4f}, mean={np.mean(lon_v):.4f}")
print(f"- Pontos únicos (aprox): {len(np.unique(np.column_stack((lat_v.round(3), lon_v.round(3))), axis=0))}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter mapa
scatter = axes[0].scatter(lon_v, lat_v, c=temp_v, s=3, alpha=0.4, cmap="RdYlBu_r", edgecolors="none")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title("Distribuição espacial com temperatura (cor)")
cbar = fig.colorbar(scatter, ax=axes[0])
cbar.set_label("temperature_C")
axes[0].grid(True, alpha=0.25)

# Distribuição marginal
axes[1].hist2d(lon_v, lat_v, bins=30, cmap="YlOrRd", cmin=1)
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_title("Densidade espacial (contagem)")

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_geoespacial.png", dpi=150)
plt.close(fig)
print("\nFigura:", FIG_DIR / "eda_geoespacial.png")

# Média de temperatura por região (lat/lon binada)
lat_bins = np.linspace(np.nanmin(lat_v), np.nanmax(lat_v), 6)
lon_bins = np.linspace(np.nanmin(lon_v), np.nanmax(lon_v), 6)
lat_idx = np.digitize(lat_v, lat_bins)
lon_idx = np.digitize(lon_v, lon_bins)

region_temps = {}
for li in np.unique(lat_idx):
    for lo in np.unique(lon_idx):
        mask = (lat_idx == li) & (lon_idx == lo)
        if mask.sum() > 0:
            region_temps[(li, lo)] = np.mean(temp_v[mask])

print("\nMédia de temperatura por região (lat x lon bins):")
for (li, lo), t_mean in sorted(region_temps.items()):
    print(f"  Lat bin {li}, Lon bin {lo}: {t_mean:.2f}°C")


Estatísticas geoespaciais:
- Latitude: min=9.1520, max=34.1988, mean=24.2499
- Longitude: min=71.3967, max=94.9120, mean=81.4705
- Pontos únicos (aprox): 76

Figura: /home/jovyan/work/notebooks/figuras/eda_geoespacial.png

Média de temperatura por região (lat x lon bins):
  Lat bin 1, Lon bin 1: 21.72°C
  Lat bin 1, Lon bin 2: 24.19°C
  Lat bin 1, Lon bin 5: 27.19°C
  Lat bin 2, Lon bin 1: 24.31°C
  Lat bin 2, Lon bin 2: 25.96°C
  Lat bin 2, Lon bin 3: 24.58°C
  Lat bin 3, Lon bin 1: 26.55°C
  Lat bin 3, Lon bin 2: 26.14°C
  Lat bin 3, Lon bin 3: 25.17°C
  Lat bin 3, Lon bin 4: 25.69°C
  Lat bin 3, Lon bin 5: 20.15°C
  Lat bin 4, Lon bin 1: 25.71°C
  Lat bin 4, Lon bin 2: 24.74°C
  Lat bin 4, Lon bin 3: 25.29°C
  Lat bin 4, Lon bin 4: 20.04°C
  Lat bin 4, Lon bin 5: 20.22°C
  Lat bin 4, Lon bin 6: 23.38°C
  Lat bin 5, Lon bin 1: 21.54°C
  Lat bin 5, Lon bin 2: 20.50°C
  Lat bin 6, Lon bin 1: 11.96°C


## Análise de outliers (temperatura por hora/mês)

Box plots para entender variabilidade intra-estrato e identificar outliers sistemáticos.


In [11]:
temp = col_to_numpy(table.column("temperature_C"), dtype=np.float64)
hour = col_to_numpy(table.column("hour"), dtype=np.float64)
month = col_to_numpy(table.column("month"), dtype=np.float64)

# Agrupa temperatura por hora
temp_by_hour = [[] for _ in range(24)]
temp_by_month = [[] for _ in range(12)]

for i in range(len(temp)):
    if not np.isnan(temp[i]):
        if not np.isnan(hour[i]) and 0 <= hour[i] < 24:
            temp_by_hour[int(hour[i])].append(temp[i])
        if not np.isnan(month[i]) and 1 <= month[i] <= 12:
            temp_by_month[int(month[i]) - 1].append(temp[i])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Box plot por hora
bp_h = axes[0].boxplot(
    temp_by_hour,
    positions=range(24),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
)
for patch in bp_h["boxes"]:
    patch.set_facecolor("#4C72B0")
    patch.set_alpha(0.7)
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("temperature_C")
axes[0].set_title("Distribuição de temperatura por hora (box plot; outliers mostrados)")
axes[0].set_xticks(range(0, 24, 2))
axes[0].grid(True, axis="y", alpha=0.3)

# Box plot por mês
bp_m = axes[1].boxplot(
    temp_by_month,
    positions=range(1, 13),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
)
for patch in bp_m["boxes"]:
    patch.set_facecolor("#C44E52")
    patch.set_alpha(0.7)
axes[1].set_xlabel("Mês")
axes[1].set_ylabel("temperature_C")
axes[1].set_title("Distribuição de temperatura por mês (box plot; outliers mostrados)")
axes[1].set_xticks(range(1, 13))
axes[1].grid(True, axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_outliers_boxplot.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_outliers_boxplot.png")

# Resumo de outliers por hour
print("\nOutliers (valores fora de 1.5*IQR) por hora:")
for h, temps in enumerate(temp_by_hour):
    if len(temps) > 0:
        q1, q3 = np.percentile(temps, [25, 75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = sum(1 for t in temps if t < lower or t > upper)
        if outliers > 0:
            print(f"  Hora {h:2d}: {outliers} outliers (range: [{lower:.2f}, {upper:.2f}])")


Figura: /home/jovyan/work/notebooks/figuras/eda_outliers_boxplot.png

Outliers (valores fora de 1.5*IQR) por hora:
  Hora  0: 8793 outliers (range: [2.40, 40.00])
  Hora  1: 7897 outliers (range: [1.75, 39.75])
  Hora  2: 7038 outliers (range: [1.05, 39.85])
  Hora  3: 6355 outliers (range: [0.50, 39.70])
  Hora  4: 5944 outliers (range: [0.05, 39.65])
  Hora  5: 5232 outliers (range: [-0.55, 39.85])
  Hora  6: 4994 outliers (range: [-0.75, 40.45])
  Hora  7: 6259 outliers (range: [0.75, 41.15])
  Hora  8: 12271 outliers (range: [5.25, 40.85])
  Hora  9: 15442 outliers (range: [8.65, 41.05])
  Hora 10: 16918 outliers (range: [10.70, 41.90])
  Hora 11: 18545 outliers (range: [11.95, 42.75])
  Hora 12: 18684 outliers (range: [12.40, 43.60])
  Hora 13: 19445 outliers (range: [12.70, 43.90])
  Hora 14: 20101 outliers (range: [12.55, 44.15])
  Hora 15: 20180 outliers (range: [12.00, 44.00])
  Hora 16: 19833 outliers (range: [10.90, 43.70])
  Hora 17: 17536 outliers (range: [9.15, 43.15])
  

## Análise de rain_label

Exploração da variável alvo original (ou feature correlata). Tipo (binária/multiclasse), distribuição e relação com `precip_mm`.


In [12]:
rain_label = table.column("rain_label")
precip_mm = col_to_numpy(table.column("precip_mm"), dtype=np.float64)

# Tipo de dados
rain_type = pc.cast(rain_label, pa.string())
rain_values = col_to_numpy(rain_type, dtype=str)

print("Valores únicos em rain_label:")
vc = pc.value_counts(rain_label)
rain_vals_vc = col_to_numpy(vc.field(0), dtype=str)
rain_cnts_vc = col_to_numpy(vc.field(1), dtype=np.int64)

for val, cnt in sorted(zip(rain_vals_vc, rain_cnts_vc), key=lambda x: x[1], reverse=True):
    pct = 100.0 * cnt / len(rain_values)
    print(f"  {val:15s}: {cnt:>10,d} ({pct:>5.2f}%)")

# Relação com precip_mm
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Gráfico de barras
unique_labels = sorted(np.unique(rain_vals_vc[~np.isin(rain_vals_vc, ['nan', 'None'])]))
label_counts = {lbl: 0 for lbl in unique_labels}
for v in rain_vals_vc:
    if v in label_counts:
        idx = list(rain_vals_vc).index(v)
        label_counts[v] = int(rain_cnts_vc[idx])

axes[0].bar(range(len(label_counts)), list(label_counts.values()), color="#55A868", edgecolor="#1a3d22", linewidth=0.65)
axes[0].set_xticks(range(len(label_counts)))
axes[0].set_xticklabels(list(label_counts.keys()), rotation=45, ha="right")
axes[0].set_ylabel("Contagem")
axes[0].set_title("Distribuição de rain_label (amostra)")
axes[0].grid(True, axis="y", alpha=0.3)

# Scatter rain_label vs precip_mm
rng = np.random.default_rng(42)
valid_precip = (~np.isnan(precip_mm)) & (rain_values != 'nan')
idx = np.flatnonzero(valid_precip)
n_plot = min(50_000, idx.shape[0])
pick = rng.choice(idx, size=n_plot, replace=False)

precip_plot = precip_mm[pick]
rain_plot = rain_values[pick]

# Mapeamento de labels para números para scatter
label_map = {lbl: i for i, lbl in enumerate(sorted(np.unique(rain_plot)))}
rain_numeric = np.array([label_map[r] for r in rain_plot])

axes[1].scatter(precip_plot, rain_numeric + np.random.normal(0, 0.05, len(rain_numeric)), 
                s=5, alpha=0.25, color="#4C72B0", edgecolors="none")
axes[1].set_xlabel("precip_mm")
axes[1].set_ylabel("rain_label")
axes[1].set_yticks(range(len(label_map)))
axes[1].set_yticklabels(sorted(label_map.keys()))
axes[1].set_title("rain_label vs precip_mm (subsample com jitter)")
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_rain_label.png", dpi=150)
plt.close(fig)
print("\nFigura:", FIG_DIR / "eda_rain_label.png")

# Estatísticas de precip_mm por label
print("\nEstatísticas de precip_mm por rain_label:")
for lbl in sorted(label_map.keys()):
    mask = rain_values == lbl
    precip_lbl = precip_mm[mask]
    precip_lbl_valid = precip_lbl[~np.isnan(precip_lbl)]
    if len(precip_lbl_valid) > 0:
        print(f"  {lbl:15s}: mean={np.mean(precip_lbl_valid):.3f}, median={np.median(precip_lbl_valid):.3f}, max={np.max(precip_lbl_valid):.3f}")


Valores únicos em rain_label:
  0              : 18,356,900 (91.78%)
  1              :  1,643,100 ( 8.22%)

Figura: /home/jovyan/work/notebooks/figuras/eda_rain_label.png

Estatísticas de precip_mm por rain_label:
  0              : mean=0.023, median=0.000, max=0.400
  1              : mean=1.731, median=1.100, max=71.600


## Padrão temporal de solar_radiation_Wm2

Verificar se tem ciclo claro por hora (esperado: 0 à noite, pico ao meio-dia). Indicador de qualidade de dados.


In [13]:
solar = col_to_numpy(table.column("solar_radiation_Wm2"), dtype=np.float64)
hour = col_to_numpy(table.column("hour"), dtype=np.float64)

# Agrupa radiação por hora
solar_by_hour = [[] for _ in range(24)]
for i in range(len(solar)):
    if not np.isnan(solar[i]) and not np.isnan(hour[i]) and 0 <= hour[i] < 24:
        solar_by_hour[int(hour[i])].append(solar[i])

# Calcula estatísticas por hora
solar_mean_h = []
solar_std_h = []
solar_p95_h = []
for h in range(24):
    if len(solar_by_hour[h]) > 0:
        solar_mean_h.append(np.mean(solar_by_hour[h]))
        solar_std_h.append(np.std(solar_by_hour[h]))
        solar_p95_h.append(np.percentile(solar_by_hour[h], 95))
    else:
        solar_mean_h.append(0)
        solar_std_h.append(0)
        solar_p95_h.append(0)

solar_mean_h = np.array(solar_mean_h)
solar_std_h = np.array(solar_std_h)
solar_p95_h = np.array(solar_p95_h)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Média e std por hora
hours_x = np.arange(24)
axes[0].plot(hours_x, solar_mean_h, color="#4C72B0", linewidth=2, marker="o", markersize=4, label="mean")
axes[0].fill_between(hours_x, solar_mean_h - solar_std_h, solar_mean_h + solar_std_h, 
                      alpha=0.25, color="#4C72B0")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("solar_radiation_Wm2")
axes[0].set_title("Ciclo diário médio de radiação solar (±1σ)")
axes[0].set_xticks(np.arange(0, 24, 2))
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Percentis por hora
axes[1].plot(hours_x, solar_mean_h, color="#4C72B0", linewidth=2, label="mean", marker="o", markersize=4)
axes[1].plot(hours_x, solar_p95_h, color="#C44E52", linewidth=1.5, linestyle="--", label="p95")
axes[1].scatter(hours_x, solar_mean_h, s=25, color="#2b4c7e", zorder=3)
axes[1].set_xlabel("Hora")
axes[1].set_ylabel("solar_radiation_Wm2")
axes[1].set_title("Média vs P95 de radiação solar por hora")
axes[1].set_xticks(np.arange(0, 24, 2))
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_solar_radiation_diaria.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_solar_radiation_diaria.png")

# Resumo
print("\nPadrão de solar_radiation_Wm2 por hora:")
print("Hora | Média     | Std       | P95")
print("-" * 45)
for h in range(24):
    print(f"{h:4d} | {solar_mean_h[h]:9.2f} | {solar_std_h[h]:9.2f} | {solar_p95_h[h]:9.2f}")

# Verificar se tem zeros noturnos
night_hours = [h for h in range(24) if h < 6 or h > 18]
day_hours = [h for h in range(6, 19)]
night_mean = np.mean([solar_mean_h[h] for h in night_hours])
day_mean = np.mean([solar_mean_h[h] for h in day_hours])
print(f"\nMédia noturna (h<6 ou h>18): {night_mean:.2f} W/m²")
print(f"Média diurna (6≤h≤18): {day_mean:.2f} W/m²")
print(f"Razão dia/noite: {day_mean/max(night_mean, 1e-6):.1f}x")


Figura: /home/jovyan/work/notebooks/figuras/eda_solar_radiation_diaria.png

Padrão de solar_radiation_Wm2 por hora:
Hora | Média     | Std       | P95
---------------------------------------------
   0 |      0.00 |      0.00 |      0.00
   1 |      0.00 |      0.00 |      0.00
   2 |      0.00 |      0.00 |      0.00
   3 |      0.00 |      0.00 |      0.00
   4 |      0.00 |      0.03 |      0.00
   5 |      1.58 |      5.71 |     10.00
   6 |     24.03 |     36.80 |    105.00
   7 |    112.61 |     88.95 |    284.00
   8 |    268.55 |    127.84 |    488.00
   9 |    437.53 |    154.66 |    674.00
  10 |    578.27 |    176.41 |    823.00
  11 |    668.96 |    191.09 |    926.00
  12 |    698.29 |    197.23 |    969.00
  13 |    667.05 |    194.98 |    951.00
  14 |    580.52 |    184.78 |    870.00
  15 |    449.21 |    166.61 |    728.00
  16 |    290.91 |    141.29 |    541.00
  17 |    137.30 |    102.21 |    331.00
  18 |     36.16 |     47.50 |    142.00
  19 |      3.14 |      

## Autocorrelação temporal (lag plots)

Verifica se température(t) correlaciona com température(t-1), (t-24), etc. Importante para feature engineering de lags e validação de estratégia temporal.


In [14]:
temp = col_to_numpy(table.column("temperature_C"), dtype=np.float64)
datetime_arr = table.column("datetime")

# Ordena por datetime (importante!)
datetime_np = col_to_numpy(datetime_arr, dtype="datetime64[ns]")
order = np.argsort(datetime_np)
temp_sorted = temp[order]
datetime_sorted = datetime_np[order]

# Remove NaN
valid_idx = ~np.isnan(temp_sorted)
temp_clean = temp_sorted[valid_idx]
datetime_clean = datetime_sorted[valid_idx]

# Calcula lags e correlação (limita a primeiros 100k pontos para performance)
max_n = min(100_000, len(temp_clean))
temp_for_lag = temp_clean[:max_n]

lags = [1, 3, 6, 12, 24, 48, 168, 672]  # 1h, 3h, 6h, 12h, 24h, 2d, 7d, 28d
autocorrs = []

print("Autocorrelação de temperature_C em diferentes lags:")
print("Lag (horas) | Correlação")
print("-" * 28)
for lag in lags:
    if lag < len(temp_for_lag):
        corr = float(np.corrcoef(temp_for_lag[:-lag], temp_for_lag[lag:])[0, 1])
        autocorrs.append(corr)
        lag_str = f"{lag}h"
        if lag == 24:
            lag_str += " (1d)"
        elif lag == 168:
            lag_str += " (7d)"
        elif lag == 672:
            lag_str += " (28d)"
        print(f"{lag_str:11s} | {corr:+.4f}")

# Plots de lag
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Lag 1 (próxima hora)
axes[0, 0].scatter(temp_for_lag[:-1], temp_for_lag[1:], s=3, alpha=0.15, color="#4C72B0", edgecolors="none")
axes[0, 0].set_xlabel("temperature_C(t)")
axes[0, 0].set_ylabel("temperature_C(t+1h)")
axes[0, 0].set_title(f"Lag-1h (corr={autocorrs[0]:+.4f})")
axes[0, 0].grid(True, alpha=0.25)

# Lag 24 (próximo dia)
axes[0, 1].scatter(temp_for_lag[:-24], temp_for_lag[24:], s=3, alpha=0.15, color="#55A868", edgecolors="none")
axes[0, 1].set_xlabel("temperature_C(t)")
axes[0, 1].set_ylabel("temperature_C(t+24h)")
axes[0, 1].set_title(f"Lag-24h (corr={autocorrs[4]:+.4f})")
axes[0, 1].grid(True, alpha=0.25)

# Lag 168 (próxima semana)
if len(autocorrs) > 6:
    axes[1, 0].scatter(temp_for_lag[:-168], temp_for_lag[168:], s=3, alpha=0.15, color="#C44E52", edgecolors="none")
    axes[1, 0].set_xlabel("temperature_C(t)")
    axes[1, 0].set_ylabel("temperature_C(t+168h)")
    axes[1, 0].set_title(f"Lag-168h=7d (corr={autocorrs[6]:+.4f})")
    axes[1, 0].grid(True, alpha=0.25)

# Gráfico de correlação vs lag
axes[1, 1].plot(lags[:len(autocorrs)], autocorrs, color="#1f2d3d", linewidth=2, marker="o", markersize=6)
axes[1, 1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1, 1].set_xlabel("Lag (horas)")
axes[1, 1].set_ylabel("Correlação")
axes[1, 1].set_title("Autocorrelação vs lag")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xscale("log")

fig.tight_layout()
fig.savefig(FIG_DIR / "eda_autocorrelacao_temporal.png", dpi=150)
plt.close(fig)
print("\nFigura:", FIG_DIR / "eda_autocorrelacao_temporal.png")

print("\n✓ Insight: Alta autocorrelação nos lags curtos (1h-6h) indica")
print("  que features de lag podem ser úteis. Se autocorr cai rapidamente,")
print("  a série é menos previsível a long-term.")


Autocorrelação de temperature_C em diferentes lags:
Lag (horas) | Correlação
----------------------------
1h          | +0.3130
3h          | +0.3123
6h          | +0.3122
12h         | +0.3129
24h (1d)    | +0.3149
48h         | +0.3091
168h (7d)   | +0.2628
672h (28d)  | -0.1460

Figura: /home/jovyan/work/notebooks/figuras/eda_autocorrelacao_temporal.png

✓ Insight: Alta autocorrelação nos lags curtos (1h-6h) indica
  que features de lag podem ser úteis. Se autocorr cai rapidamente,
  a série é menos previsível a long-term.


## Figuras exportadas (caminhos)

Relativos à raiz do repositório:

- `notebooks/figuras/eda_temperature_distribuicao.png`
- `notebooks/figuras/eda_temperature_por_hour_month.png`
- `notebooks/figuras/eda_temperature_scatter_relacoes.png`
- `notebooks/figuras/eda_correlacao_preditoras.png`
- `notebooks/figuras/eda_hour_month_histogramas.png`
- `notebooks/figuras/eda_contagem_por_dia.png`
- `notebooks/figuras/eda_completude_dados.png`
- `notebooks/figuras/eda_geoespacial.png`
- `notebooks/figuras/eda_outliers_boxplot.png`
- `notebooks/figuras/eda_rain_label.png`
- `notebooks/figuras/eda_solar_radiation_diaria.png`
- `notebooks/figuras/eda_autocorrelacao_temporal.png`


## Teste: Filtro de temperaturas negativas

Remove linhas com `temperature_C < 0` e analisa o impacto (quantidade de linhas, % descartadas).


In [ ]:
# Estado inicial
n_inicial = table.num_rows
temp_col = table.column("temperature_C")

# Filtra: temperatura >= 0
mask_valid_temp = pc.greater_equal(temp_col, 0)
table_filtered = table.filter(mask_valid_temp)

n_final = table_filtered.num_rows
n_removido = n_inicial - n_final
pct_removido = 100.0 * n_removido / n_inicial if n_inicial > 0 else 0

print("=" * 60)
print("FILTRO: temperature_C >= 0")
print("=" * 60)
print(f"Linhas iniciais:  {n_inicial:>12,d}")
print(f"Linhas removidas: {n_removido:>12,d} ({pct_removido:.2f}%)")
print(f"Linhas finais:    {n_final:>12,d}")
print("=" * 60)

# Analisa as removidas
if n_removido > 0:
    temp_removed = col_to_numpy(table.filter(pc.invert(mask_valid_temp)).column("temperature_C"), dtype=np.float64)
    temp_removed_valid = temp_removed[~np.isnan(temp_removed)]
    if len(temp_removed_valid) > 0:
        print(f"\nTemperaturas removidas (< 0°C):")
        print(f"  Min:    {np.min(temp_removed_valid):>8.2f}°C")
        print(f"  Max:    {np.max(temp_removed_valid):>8.2f}°C")
        print(f"  Média:  {np.mean(temp_removed_valid):>8.2f}°C")
        print(f"  Contagem: {len(temp_removed_valid):>6,d}")

# Comparação de distribuição (antes vs depois)
temp_before = col_to_numpy(table.column("temperature_C"), dtype=np.float64)
temp_after = col_to_numpy(table_filtered.column("temperature_C"), dtype=np.float64)

temp_before_valid = temp_before[~np.isnan(temp_before)]
temp_after_valid = temp_after[~np.isnan(temp_after)]

print(f"\nEstatísticas de temperature_C:")
print(f"{'':20s} | {'Antes':>12s} | {'Depois':>12s}")
print("-" * 48)
print(f"{'Min':20s} | {np.min(temp_before_valid):>12.2f} | {np.min(temp_after_valid):>12.2f}")
print(f"{'P05':20s} | {np.percentile(temp_before_valid, 5):>12.2f} | {np.percentile(temp_after_valid, 5):>12.2f}")
print(f"{'Média':20s} | {np.mean(temp_before_valid):>12.2f} | {np.mean(temp_after_valid):>12.2f}")
print(f"{'Mediana':20s} | {np.median(temp_before_valid):>12.2f} | {np.median(temp_after_valid):>12.2f}")
print(f"{'P95':20s} | {np.percentile(temp_before_valid, 95):>12.2f} | {np.percentile(temp_after_valid, 95):>12.2f}")
print(f"{'Max':20s} | {np.max(temp_before_valid):>12.2f} | {np.max(temp_after_valid):>12.2f}")
